# SentinelMail — Ensemble: Weight Learning & Evaluation

Confidence-weighted **per-category late fusion** of the four models
(BERT, RoBERTa, Bi-LSTM, rule-based). This notebook fits per-label fusion
weights on a **DEV** half of the validation set and reports on a **held-out
EVAL** half, so the ensemble gets no peek at the rows it is scored on
(avoids tuning-on-test bias). Single models are re-evaluated on the same EVAL
half for a fair RQ1 comparison.

**Prerequisite:** all four model checkpoints must exist under `models/*/checkpoint/`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

RESULTS_DIR = REPO_ROOT / "evaluation" / "results"
ENSEMBLE_DIR = REPO_ROOT / "ensemble"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
THRESHOLD = 0.5
print(f"Device: {DEVICE}")

In [ ]:
from ensemble import (
    build_cache,
    load_probas,
    fit_weights,
    tune_thresholds,
    save_weights,
    save_thresholds,
)
from evaluation import (
    LABEL_COLS,
    compute_all_metrics,
    fuse,
    run_ablation,
    plot_ablation_table,
    plot_model_comparison_table,
    plot_confusion_matrices,
    plot_per_label_bars,
    plot_roc_curves,
    plot_pr_curves,
    save_results,
)

## 1. Build / Load Probability Cache

`build_cache` runs each detector's `predict_proba` once over the validation
split and caches the arrays (BERT/RoBERTa CPU inference is the bottleneck). It
is idempotent — re-running is instant unless the split text changes. The loader
verifies an sha1 of the ordered emails so every model's rows are aligned.

In [ ]:
build_cache(split="validation", device=str(DEVICE))
model_probas, y_true, texts = load_probas(split="validation")

N = y_true.shape[0]
print(f"Loaded {N:,} emails; models: {list(model_probas)}")
for name, arr in model_probas.items():
    print(f"  {name:12s} {arr.shape}  range [{arr.min():.3f}, {arr.max():.3f}]")

## 2. DEV / EVAL Split

Deterministic 50/50 split of the validation set. Weights and thresholds are fit
on **DEV**; all reported metrics (ensemble and single models) come from the
held-out **EVAL** half.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(N)
dev_idx, eval_idx = perm[: N // 2], perm[N // 2 :]

dev_probas  = {m: p[dev_idx]  for m, p in model_probas.items()}
eval_probas = {m: p[eval_idx] for m, p in model_probas.items()}
dev_y,  eval_y  = y_true[dev_idx], y_true[eval_idx]
eval_texts = [texts[i] for i in eval_idx]

print(f"DEV: {len(dev_idx):,}   EVAL: {len(eval_idx):,}")

## 3. Fit Per-Category Weights (on DEV)

Per-label grid + Nelder-Mead search on the **true** binary F1. Fusion is
separable per label, so each label's four model-weights are optimised
independently. (`fit_weights_lbfgsb` is available as the spec-letter surrogate
alternative — see `ensemble/train_weights.py`.)

In [ ]:
weights = fit_weights(dev_probas, dev_y, threshold=THRESHOLD)
thresholds = tune_thresholds(dev_probas, dev_y, weights)

save_weights(weights, ENSEMBLE_DIR / "weights.json")
save_thresholds(thresholds, ENSEMBLE_DIR / "thresholds.json")
print("Saved weights.json and thresholds.json")
print("Tuned per-label thresholds:", thresholds)

### 3a. Learned Weights (RQ2 — does complementarity vary by category?)

In [ ]:
weights_df = pd.DataFrame(weights).T[LABEL_COLS].round(3)  # rows=model, cols=label
weights_df

## 4. Evaluate Ensemble (held-out EVAL)

In [ ]:
y_pred_e, y_proba_e = fuse(eval_probas, weights, threshold=THRESHOLD)
metrics = compute_all_metrics(eval_y, y_pred_e, y_proba=y_proba_e)

ci = metrics["macro_f1_ci"]
print(f"Ensemble macro-F1: {metrics['macro_f1']:.4f}  "
      f"95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
for label in LABEL_COLS:
    print(f"  {label:12s} F1={metrics['per_label'][label]['f1']:.4f}")

### 4a. Threshold Sensitivity (0.3 / 0.5 / 0.7 + tuned per-label)

In [ ]:
rows = []
for t in [0.3, 0.5, 0.7]:
    m = compute_all_metrics(eval_y, (y_proba_e >= t).astype(np.int32), n_bootstrap=50)
    row = {"config": f"fixed@{t}", "macro_f1": round(m["macro_f1"], 4)}
    for label in LABEL_COLS:
        row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
    rows.append(row)

# tuned per-label thresholds
y_pred_tuned = np.stack(
    [(y_proba_e[:, i] >= thresholds[label]).astype(np.int32)
     for i, label in enumerate(LABEL_COLS)], axis=1)
m = compute_all_metrics(eval_y, y_pred_tuned, n_bootstrap=50)
row = {"config": "tuned_per_label", "macro_f1": round(m["macro_f1"], 4)}
for label in LABEL_COLS:
    row[f"{label}_f1"] = round(m["per_label"][label]["f1"], 4)
rows.append(row)

pd.DataFrame(rows).set_index("config")

## 5. Ablation Study (RQ1 — does the ensemble beat any single model?)

`run_ablation` reports the full ensemble, each leave-one-model-out
configuration, every solo model, and the best single model — all on the EVAL
half with 95% bootstrap CIs.

In [ ]:
ablation_df = run_ablation(eval_probas, weights, eval_y, threshold=THRESHOLD)
plot_ablation_table(ablation_df)

## 6. Model Comparison (recomputed on the same EVAL half)

Single-model metrics are recomputed here from the cached EVAL-half probabilities
at threshold 0.5 — **not** loaded from the full-validation JSONs, which would be
an unfair comparison against the held-out ensemble.

In [ ]:
results = {"ensemble": metrics}
for name, probas in eval_probas.items():
    y_pred_m = (probas >= THRESHOLD).astype(np.int32)
    # rule_based produces binary 0/1 flags, not calibrated probabilities -> y_proba=None
    y_proba_m = None if name == "rule_based" else probas
    results[name] = compute_all_metrics(eval_y, y_pred_m, y_proba=y_proba_m)

plot_model_comparison_table(results)

## 7. Confusion Matrices, Per-Label Bars, ROC & PR Curves

In [ ]:
_, fig = plot_confusion_matrices(eval_y, y_pred_e, normalize="true", return_fig=True)
fig

In [ ]:
_, fig = plot_per_label_bars(metrics, return_fig=True)
fig

In [ ]:
roc_data = plot_roc_curves(eval_y, y_proba_e)
pr_data = plot_pr_curves(eval_y, y_proba_e)

## 8. Save Results

In [ ]:
out_path = RESULTS_DIR / "ensemble_metrics.json"

save_results(
    metrics,
    path=out_path,
    model_name="ensemble_weighted_fusion",
    n_samples=len(eval_idx),
    threshold=THRESHOLD,
    extra_metadata={
        "fusion": "per-category weighted late fusion",
        "optimizer": "per-label grid + Nelder-Mead on true F1",
        "eval_split": "held_out_half",
        "random_state": RANDOM_STATE,
        "weights": weights,
        "thresholds": thresholds,
    },
)
print(f"Saved to {out_path}")